# Bài tập buổi 6 — Hồi quy tuyến tính

**Sinh viên thực hiện:** "Nguyễn Lê Gia Bảo"

---

## Bối cảnh

Trong bài tập này, bạn sẽ làm việc với bộ dữ liệu **California Housing** (Dự đoán giá nhà tại California dựa trên các đặc trưng nhân khẩu học và địa lý). Bài toán đặt ra là bài toán **Hồi quy (Regression)**.

Nhiệm vụ của bạn là xây dựng luồng xử lý dữ liệu và huấn luyện 3 biến thể của Hồi quy tuyến tính:
1. **Vanilla Linear Regression** (Hồi quy tuyến tính thông thường)
2. **Ridge Regression** (Hồi quy với chuẩn hóa L2)
3. **Lasso Regression** (Hồi quy với chuẩn hóa L1)

## Mục tiêu bài tập

1. Thực hiện Load dữ liệu và Khám phá dữ liệu (EDA) cơ bản.
2. Chia tập Train/Test và tiền xử lý (Scaling) đúng chuẩn, không gây Data Leakage.
3. Huấn luyện và đánh giá mô hình bằng các metric chuẩn cho Regression (RMSE, R²).
4. **Trực quan hóa hệ số hồi quy (Coefficients)** để hiểu rõ tính chất thu nhỏ (shrinkage) của Ridge và khả năng chọn lọc đặc trưng (feature selection) của Lasso.

---


## 0. Chuẩn bị môi trường & Import Thư viện

Ô này chứa sẵn các thư viện cần thiết. Nếu bạn cần dùng thêm thư viện nào, hãy bổ sung vào đây.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)  # Cố định random seed
print("Đã import xong thư viện.")

---
## Task 1 — Tải dữ liệu và Khám phá ban đầu (EDA)

### Yêu cầu
1. Tải bộ dữ liệu `California Housing` từ `sklearn.datasets`.
2. Chuyển đổi thành pandas DataFrame. Gán cột target là `MedHouseVal` (Giá nhà trung bình - đơn vị trăm nghìn USD).
3. In ra số dòng, số cột (shape) và thông tin kiểu dữ liệu (`.info()`).
4. Kiểm tra xem có giá trị thiếu (Missing values) nào trong bộ dữ liệu không.

### Gợi ý
- Hàm `fetch_california_housing(as_frame=True)` hỗ trợ trả về DataFrame trực tiếp qua thuộc tính `.frame`.
- Dùng `df.isnull().sum()` để đếm số lượng giá trị thiếu cho mỗi cột.

In [ ]:
california = fetch_california_housing(as_frame=True)
df = california.frame

print("Shape của dữ liệu:", df.shape)
print()
df.info()
print()
print("Số giá trị thiếu theo từng cột:")
print(df.isnull().sum())
print()
print("Tổng số giá trị thiếu:", df.isnull().sum().sum())
print()
df.head()

---
## Task 2 — Trực quan hóa Phân phối & Tương quan

### Yêu cầu
1. Vẽ biểu đồ Histogram (có đường cong KDE) cho biến mục tiêu `MedHouseVal`. Nhận xét xem phân phối có bị lệch (skew) không.
2. Vẽ ma trận tương quan (Heatmap) giữa tất cả các biến số trong dữ liệu.
3. **Trả lời:** Đặc trưng nào có độ tương quan dương mạnh nhất với giá nhà (`MedHouseVal`)?

### Gợi ý
- Sử dụng `sns.histplot(data=df, x='MedHouseVal', kde=True)` để vẽ phân phối.
- Dùng `df.corr()` để tính ma trận tương quan và đưa vào `sns.heatmap(..., annot=True, cmap='coolwarm', fmt=".2f")` để vẽ biểu đồ nhiệt.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x='MedHouseVal', kde=True, color='steelblue', ax=ax)
ax.set_title("Phân phối của MedHouseVal (Giá nhà trung bình)")
ax.set_xlabel("MedHouseVal (trăm nghìn USD)")
ax.set_ylabel("Số lượng")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Ma trận tương quan giữa các đặc trưng")
plt.tight_layout()
plt.show()

print("Tương quan của từng đặc trưng với MedHouseVal (sắp xếp giảm dần):")
print(corr_matrix['MedHouseVal'].sort_values(ascending=False))

**Trả lời 2:**

**Nhận xét về phân phối của MedHouseVal:**

Phân phối của biến mục tiêu `MedHouseVal` bị **lệch phải (right-skewed / positive skew)**. Cụ thể:
- Phân phối có đuôi dài về phía bên phải, cho thấy tồn tại một số căn nhà có giá rất cao so với số đông.
- Giá trị tập trung nhiều ở khoảng 1–3 (trăm nghìn USD), với đỉnh (mode) khoảng 1.5.
- Đặc biệt có một **cụm điểm dày đặc tại giá trị 5.0**, đây là giá trị bị cắt ngưỡng (capped) trong bộ dữ liệu gốc — tức tất cả giá nhà trên 500,000 USD đều được gán bằng 5.0.
- Sự lệch này có thể ảnh hưởng đến hiệu suất của mô hình hồi quy tuyến tính vì giả định phân phối chuẩn của phần dư.

**Đặc trưng có tương quan dương mạnh nhất với MedHouseVal:**

`MedInc` (Thu nhập trung bình của hộ gia đình) là đặc trưng có **tương quan dương mạnh nhất** với `MedHouseVal`, với hệ số tương quan Pearson xấp xỉ **+0.69**. Điều này hoàn toàn hợp lý về mặt kinh tế: khu vực có thu nhập bình quân cao thường đi kèm với giá bất động sản cao hơn.


---
## Task 3 — Chia tập dữ liệu và Tiền xử lý (Scaling)

Với các mô hình có sử dụng Regularization như Ridge và Lasso, việc **Scale dữ liệu** (đưa về cùng thang đo) là **BẮT BUỘC**. Nếu không scale, các đặc trưng có miền giá trị lớn (ví dụ như Population) sẽ lấn át các đặc trưng có miền giá trị nhỏ, làm sai lệch tác dụng của hệ số phạt L1/L2.

### Yêu cầu
1. Tách features (`X`) và target (`y`).
2. Chia tập Train/Test theo tỷ lệ **80/20**.
3. Dùng `StandardScaler` để scale tập `X`. **Quan trọng:** Chỉ `.fit()` trên `X_train`, sau đó dùng tham số đã học đó để `.transform()` cho cả `X_train` và `X_test`.

### Gợi ý
- `train_test_split(X, y, test_size=0.2, random_state=42)`.
- `scaler.fit_transform(X_train)` cho tập train và `scaler.transform(X_test)` cho tập test.

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Kích thước X_train: {X_train_scaled.shape}")
print(f"Kích thước X_test:  {X_test_scaled.shape}")
print(f"Kích thước y_train: {y_train.shape}")
print(f"Kích thước y_test:  {y_test.shape}")

---
## Task 4 — Huấn luyện Vanilla Linear Regression

### Yêu cầu
1. Khởi tạo và huấn luyện mô hình `LinearRegression` trên tập Train đã scale.
2. Dự đoán trên tập Test.
3. Tính và in ra 2 chỉ số đánh giá: **RMSE** (Root Mean Squared Error) và **R² Score**.

### Gợi ý
- RMSE có thể tính bằng `np.sqrt(mean_squared_error(y_true, y_pred))`.
- Hàm `r2_score(y_true, y_pred)` sẽ trả về chỉ số R² (hệ số xác định).

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)
print(f"Linear Regression - RMSE: {rmse_lr:.4f}, R2: {r2_lr:.4f}")

---
## Task 5 — Huấn luyện Ridge và Lasso Regression

### Yêu cầu
1. Huấn luyện mô hình **Ridge** với tham số siêu việt `alpha=10.0`.
2. Huấn luyện mô hình **Lasso** với tham số siêu việt `alpha=0.1`.
3. Tính RMSE và R² cho cả 2 mô hình trên tập Test và in kết quả để so sánh.

### Gợi ý
- Khởi tạo mô hình: `Ridge(alpha=10.0)` và `Lasso(alpha=0.1)`.
- Bạn có thể viết một hàm `evaluate_model(model, X_test, y_test)` nhỏ để tái sử dụng code tính RMSE và R² cho đỡ lặp lại.

In [ ]:
def evaluate_model(model, X_test_scaled, y_test):
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    return rmse, r2

ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train_scaled, y_train)
rmse_ridge, r2_ridge = evaluate_model(ridge_model, X_test_scaled, y_test)
print(f"Ridge Regression  - RMSE: {rmse_ridge:.4f}, R2: {r2_ridge:.4f}")

lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_train)
rmse_lasso, r2_lasso = evaluate_model(lasso_model, X_test_scaled, y_test)
print(f"Lasso Regression  - RMSE: {rmse_lasso:.4f}, R2: {r2_lasso:.4f}")

print()
print("=== Bảng so sánh ===")
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge (alpha=10)', 'Lasso (alpha=0.1)'],
    'RMSE': [rmse_lr, rmse_ridge, rmse_lasso],
    'R²': [r2_lr, r2_ridge, r2_lasso]
})
print(results.to_string(index=False))

---
## Task 6 — Trực quan hóa Hệ số hồi quy (Coefficients)

Đây là phần cốt lõi để thấy sự khác biệt về mặt toán học giữa L1 và L2 Regularization.

### Yêu cầu
1. Lấy mảng hệ số hồi quy (`.coef_`) từ cả 3 mô hình (Linear, Ridge, Lasso).
2. Tạo một DataFrame lưu trữ các hệ số này với Index là tên các features.
3. Vẽ biểu đồ Barplot so sánh các trọng số của 3 mô hình cạnh nhau cho từng feature.
4. **Trả lời:** Quan sát hệ số của Lasso, bạn thấy điều gì đặc biệt xảy ra với một số features? Tính chất này thường được ứng dụng để làm gì?

### Gợi ý
- `california.feature_names` trả về danh sách tên cột.
- Có thể gom thành Pandas DataFrame và gọi lệnh `df_coefs.plot(kind='bar', figsize=(12, 6))` để pandas tự động vẽ các thanh cạnh nhau.

In [ ]:
features = california.feature_names
coef_df = pd.DataFrame({
    'Linear': lr_model.coef_,
    'Ridge': ridge_model.coef_,
    'Lasso': lasso_model.coef_
}, index=features)

print("Bảng hệ số hồi quy:")
print(coef_df.round(4))

ax = coef_df.plot(kind='bar', figsize=(12, 6), color=['#2196F3', '#FF9800', '#F44336'])
plt.title("So sánh hệ số hồi quy của các mô hình", fontsize=14)
plt.ylabel("Giá trị Coefficient")
plt.xlabel("Đặc trưng (Feature)")
plt.xticks(rotation=30, ha='right')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.legend(title='Mô hình')
plt.tight_layout()
plt.show()

print("\nCác hệ số Lasso bằng 0 (đặc trưng bị loại):")
zero_coef = coef_df[coef_df['Lasso'] == 0].index.tolist()
print(zero_coef if zero_coef else "Không có đặc trưng nào bị loại với alpha=0.1")

**Trả lời 6:**

**Nhận xét về hệ số của Lasso so với Linear và Ridge:**

Khi quan sát biểu đồ so sánh hệ số hồi quy, ta thấy sự khác biệt rõ ràng giữa ba mô hình:

1. **Linear Regression**: Hệ số có thể nhận bất kỳ giá trị nào, không bị ràng buộc. Một số hệ số có thể rất lớn hoặc rất nhỏ tùy theo dữ liệu.

2. **Ridge Regression (L2)**: Hệ số bị *co lại* (shrinkage) về phía 0 nhưng **không bao giờ bằng đúng 0**. Ridge thu nhỏ đồng đều tất cả hệ số, giúp giảm phương sai (variance) của mô hình.

3. **Lasso Regression (L1)**: Đây là điểm đặc biệt nhất — Lasso có khả năng ép một số hệ số về **đúng bằng 0**. Cụ thể với `alpha=0.1`, một số đặc trưng ít quan trọng sẽ có coefficient = 0 hoàn toàn.

**Ý nghĩa của hiện tượng hệ số bị ép về 0:**

Đây chính là tính chất **Feature Selection (Chọn lọc đặc trưng)** tự động của Lasso. Khi hệ số của một đặc trưng bị ép về 0, đặc trưng đó thực chất đã bị loại khỏi mô hình — mô hình coi đặc trưng đó không có đóng góp vào dự đoán. Tính chất này cực kỳ hữu ích trong các bài toán có **chiều dữ liệu cao (high-dimensional data)**, giúp:
- Tự động xác định các đặc trưng quan trọng
- Tạo ra mô hình thưa thớt (sparse model) dễ giải thích hơn
- Giảm nguy cơ overfitting khi có nhiều đặc trưng không liên quan


---
## (Bonus) Task 7 — Tìm siêu tham số tối ưu với GridSearchCV

Ở Task 5, chúng ta chỉ chọn bừa `alpha=10.0` và `alpha=0.1`. Làm sao để biết `alpha` bao nhiêu là tốt nhất cho bộ dữ liệu này?

### Yêu cầu
Sử dụng `GridSearchCV` để chạy thử nghiệm nghiệm chéo (Cross-Validation) tìm giá trị `alpha` tối ưu cho Ridge Regression trong danh sách: `[0.1, 1.0, 10.0, 100.0]`.

### Gợi ý
- Import: `from sklearn.model_selection import GridSearchCV`.
- Cấu hình param grid: `param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}`.
- In ra `grid_search.best_params_` sau khi fit trên tập Train.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}
grid_search = GridSearchCV(Ridge(), param_grid, cv=5, scoring='neg_root_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

print(f"Alpha tốt nhất cho Ridge: {grid_search.best_params_}")
print(f"RMSE CV tốt nhất (train): {-grid_search.best_score_:.4f}")

best_ridge = grid_search.best_estimator_
rmse_best, r2_best = evaluate_model(best_ridge, X_test_scaled, y_test)
print(f"Ridge tốt nhất trên Test - RMSE: {rmse_best:.4f}, R2: {r2_best:.4f}")

print("\nKết quả chi tiết cross-validation:")
cv_results = pd.DataFrame(grid_search.cv_results_)[['param_alpha', 'mean_test_score', 'std_test_score']]
cv_results['mean_RMSE'] = -cv_results['mean_test_score']
print(cv_results[['param_alpha', 'mean_RMSE', 'std_test_score']].to_string(index=False))

---
## Bảng tự kiểm trước khi nộp

- [x] Notebook chạy **Restart & Run All** không bị lỗi NameError hay SyntaxError.
- [x] Đã hoàn thiện bước Scale dữ liệu cẩn thận, không có Data Leakage.
- [x] Đã trực quan hóa đủ các biểu đồ ở Task 2 và Task 6.
- [x] Đã trả lời phần nhận xét bằng Text ở các câu hỏi (Task 2, Task 6).
